# Capítulo 7: O que é Aprendizado Estatístico

**Bases 5 — Ciência de Dados** · notebook de aula

Cada célula de código é a mesma do livro e roda na ordem em que aparece — execute de cima para baixo. Versão publicada deste capítulo: [https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/index.html](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/index.html)

> **Gerado automaticamente a partir dos `.qmd` do livro por `scripts/gerar-notebooks.py`.** Edições feitas aqui se perdem no próximo `make notebooks`; para mudar o conteúdo, edite o `.qmd`.

In [ ]:
# Põe o diretório de trabalho na raiz do projeto — é o que faz
# `from scratch...` e os caminhos `dados/...` funcionarem. No livro isso vem
# do `execute-dir: project` do Quarto; aqui é feito à mão.
#
# No Colab não existe cópia do projeto, então esta célula clona uma. É rápido
# (clone raso) e acontece só na primeira execução da sessão.
import os
import subprocess
import sys

REPO = "https://github.com/BragaD/UnDF-Bases5-CienciaDeDados-202602.git"


def raiz_do_projeto(inicio="."):
    """Sobe os diretórios até achar o `_quarto.yml`. None se não houver."""
    atual = os.path.abspath(inicio)
    while not os.path.exists(os.path.join(atual, "_quarto.yml")):
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai
    return atual


raiz = raiz_do_projeto()
if raiz is None:
    destino = "/content/bases5" if os.path.isdir("/content") else "bases5"
    if not os.path.isdir(destino):
        print("baixando o material da disciplina...")
        subprocess.run(["git", "clone", "--depth", "1", REPO, destino], check=True)
    raiz = raiz_do_projeto(destino)

os.chdir(raiz)
if raiz not in sys.path:
    sys.path.insert(0, raiz)

%matplotlib inline
print("diretório de trabalho:", os.getcwd())

> **📌 Nota**
>
> Este capítulo corresponde ao capítulo 2 de James et al. (2023).

> **⚠️ Atenção — Em construção**
>
> A visão geral deste capítulo ainda será escrita.

## Seções

| Seção | Tópico |
|---|---|
| [7.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/01-o-array.html) | O Array |
| [7.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/02-estimar-f.html) | Estimar f: Predição e Inferência |
| [7.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/03-parametrico-e-nao-parametrico.html) | Paramétrico e Não Paramétrico |
| [7.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/04-precisao-contra-interpretabilidade.html) | Precisão contra Interpretabilidade |
| [7.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/05-supervisionado-e-nao-supervisionado.html) | Supervisionado e Não Supervisionado |
| [7.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/06-qualidade-do-ajuste-e-vies-variancia.html) | Qualidade do Ajuste e o Compromisso Viés-Variância |
| [7.7](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/07-classificacao-e-o-classificador-de-bayes.html) | Classificação e o Classificador de Bayes |

## O Array

> **📌 Nota**
>
> Esta seção corresponde à seção 2.3 de James et al. (2023).

O capítulo anterior fechou com `X` e `y` prontos, os dois ainda como `DataFrame` e `Series` — rótulos de coluna, índice, tudo o que o `pandas` guarda sobre a tabela. A partir daqui o objeto que aparece o tempo todo é outro: todo estimador do `scikit-learn` devolve um `ndarray` — um coeficiente ajustado, uma previsão, uma probabilidade —, e ninguém apresentou esse objeto ainda. Este é o array do `numpy`: o que ele guarda, como se fatia, como se filtra, e o que significa somar "ao longo de um eixo".

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("estilo-figuras.mplstyle")

### O array, e o tipo único que ele impõe

Um `np.array` nasce de uma lista, mas não herda a flexibilidade dela: uma lista Python guarda qualquer mistura de tipos, um array guarda só um.

In [ ]:
quartos = np.array([2, 3, 1, 4, 2])
quartos, quartos.dtype

`quartos` guarda os mesmos cinco números da lista, mas ganha um atributo que a lista não tem: `dtype`, o tipo único de todo elemento do array — aqui, `int64`. Basta um elemento vir com casa decimal para promover o array inteiro:

In [ ]:
misto = np.array([1, 2, 3.0])
misto.dtype

`misto.dtype` devolve `float64`, não uma mistura de `int64` e `float64` elemento a elemento — os dois primeiros números também viraram ponto flutuante, mesmo tendo entrado como inteiros. Uma lista de listas vira um array de duas dimensões, e `shape` guarda o formato:

In [ ]:
matriz = np.array([[1, 2], [3, 4], [5, 6]])
matriz.shape, matriz.dtype

`(3, 2)` diz três linhas e duas colunas; `dtype`, de novo, é um só para o array inteiro.

### Fatiar sem copiar: a vista

Fatiar um array usa a mesma notação `[início:fim]` de uma lista Python, mas o resultado se comporta diferente.

In [ ]:
original = np.array([10, 20, 30, 40, 50])
fatia = original[1:3]
fatia[0] = 999
original

Alterar `fatia[0]` também mudou `original`. A fatia não é uma cópia dos números: é outra janela sobre o mesmo bloco de memória. É a armadilha número um de quem chega de listas, onde `lista[1:3]` sempre devolve uma lista nova, sem ligação nenhuma com a original. Quando o array de origem precisa continuar intocado, a fatia pede `.copy()`:

In [ ]:
original2 = np.array([10, 20, 30, 40, 50])
copia = original2[1:3].copy()
copia[0] = 999
original2

Desta vez `original2` sai como entrou — `.copy()` força um bloco de memória novo, separado do original.

> **🔷 Conceito**
>
> Fatia (`x[1:3]`) devolve uma **vista**: o mesmo bloco de memória, só enxergado por outra janela. Indexação por **máscara booleana** ou por **lista de posições** (`x[[0, 2]]`) sempre devolve uma **cópia**. `.copy()` força uma cópia em qualquer um dos dois casos, quando o array original precisa ficar fora de alcance.

### A máscara booleana no lugar do laço

Uma comparação entre um array e um número não compara o array inteiro de uma vez: compara elemento a elemento, e devolve um array de `True`/`False` do mesmo tamanho.

In [ ]:
valores = np.array([-3, 5, -1, 8, 0, -7, 2])
mascara = valores > 0
mascara

Usar esse array de booleanos para indexar o array original — `valores[mascara]` — filtra os elementos onde a máscara vale `True`:

In [ ]:
valores[mascara]

É o mesmo resultado que um laço `for` com um `if` dentro produziria, elemento a elemento, só que sem escrever o laço: a máscara descreve a condição uma vez, e o `numpy` aplica sobre o array inteiro.

### A forma de uma redução: `axis`

Somar os elementos de uma matriz aceita um argumento que muda o que "somar" quer dizer: `axis=0` percorre as linhas, coluna por coluna; `axis=1` percorre as colunas, linha por linha. Sobre uma matriz com aluguel e área de quatro imóveis:

In [ ]:
precos = np.array([
    [1200.0, 65.0],
    [800.0, 42.0],
    [2100.0, 98.0],
    [950.0, 55.0],
])
precos.shape

Quatro linhas, duas colunas. Somando ao longo de `axis=0`:

In [ ]:
soma_colunas = precos.sum(axis=0)
soma_colunas.shape, soma_colunas

A forma sai `(2,)` — um total por **coluna**: 5050,0 de aluguel somado, 260,0 de área somada, os quatro imóveis colapsados em uma soma cada. Somando ao longo de `axis=1`:

In [ ]:
soma_linhas = precos.sum(axis=1)
soma_linhas.shape, soma_linhas

A forma agora sai `(4,)` — o oposto: um total por **linha**, aluguel mais área de cada imóvel, os dois números de cada linha colapsados em um só. É a confusão mais comum de quem começa com `axis`: o número que ele nomeia é o eixo que **desaparece** na redução, não o eixo que sobra.

`mean` segue a mesma regra de forma que `sum` — só troca soma por média:

In [ ]:
media_colunas = precos.mean(axis=0)
media_colunas.shape, media_colunas

Forma `(2,)`, de novo uma média por **coluna**: 1262,5 de aluguel médio, 65,0 de área média.

In [ ]:
media_linhas = precos.mean(axis=1)
media_linhas.shape, media_linhas

Forma `(4,)`, uma média por **linha** — um número por imóvel. É a redução que mais volta nos capítulos seguintes: média de coluna para padronizar uma variável, média de linha para resumir uma observação.

### Sorteando com semente: o `rng`

Todo sorteio deste material usa um gerador com semente fixa e explícita, passado adiante em vez de guardado num estado global. Sem semente, cada renderização da página sortearia números diferentes — e as figuras que dependem deles mudariam a cada vez, sem que o texto ao redor mudasse junto.

In [ ]:
rng_a = np.random.default_rng(7)
rng_b = np.random.default_rng(7)
np.array_equal(rng_a.normal(size=3), rng_b.normal(size=3))

Duas instâncias criadas com a mesma semente sorteiam exatamente a mesma sequência — é essa garantia que faz `rng = np.random.default_rng(7)` valer como semente fixa do capítulo inteiro, a partir daqui:

In [ ]:
rng = np.random.default_rng(7)
amostra = rng.normal(size=5)
amostra

`rng.normal` sorteia de uma normal padrão. `rng.choice` sorteia entre valores dados, cada um com a mesma chance por padrão:

In [ ]:
rng.choice(["sim", "não"], size=5)

A figura a seguir usa as duas coisas desta seção ao mesmo tempo: duas coordenadas sorteadas com `rng.normal`, e uma máscara booleana que decide a cor de cada ponto.

In [ ]:
# Figura: Duzentos pontos sorteados com `rng.normal`, coloridos por uma máscara booleana sobre a distância à origem
x = rng.normal(size=200)
y = rng.normal(size=200)
distancia = np.sqrt(x**2 + y**2)
mascara_fig = distancia > 1.5

fig, ax = plt.subplots()
ax.scatter(x[~mascara_fig], y[~mascara_fig], label="até 1,5 da origem")
ax.scatter(x[mascara_fig], y[mascara_fig], label="além de 1,5 da origem")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
mascara_fig.sum(), (~mascara_fig).sum()

55 dos 200 pontos ficam a mais de 1,5 da origem; os outros 145 ficam mais perto — a mesma máscara que filtrou `valores` na seção anterior, agora decidindo a cor de um gráfico em vez de filtrar uma lista de números.

### Do `DataFrame` para o array

O capítulo anterior separou `alugueis` em `X`, os preditores, e `y`, o alvo — os dois como objetos do `pandas`. Aqui `X` volta reduzido às quatro colunas numéricas que já vinham prontas — sem os cinco `dummies` de cidade que a seção 6.6 acrescenta depois —, só para a linha caber numa página:

In [ ]:
alugueis = pd.read_csv("dados/alugueis.csv", na_values=["-"])
X = alugueis[["area_m2", "quartos", "banheiros", "vagas"]]
y = alugueis["aluguel"]
type(X), X.shape, type(y), y.shape

`.to_numpy()` devolve o que está por baixo de cada um: só os números, sem rótulo de coluna nem índice.

In [ ]:
X_array = X.to_numpy()
y_array = y.to_numpy()
type(X_array), X_array.shape, X_array.dtype, type(y_array), y_array.shape, y_array.dtype

Mesmas formas, `(10692, 4)` e `(10692,)`, mas o `DataFrame` virou `ndarray` e a `Series` virou `ndarray` também — e junto com o tipo foram embora os nomes de coluna e o índice. A primeira linha de `X_array`,

In [ ]:
X_array[0]

chega como quatro números soltos — 70, 2, 1, 1 — sem dizer mais qual era área, qual era quarto, qual era banheiro, qual era vaga; só a ordem em que as colunas foram escolhidas continua carregando esse significado. É o mesmo tipo de objeto, sem nome nenhum de coluna, que sai do outro lado de um estimador do `scikit-learn`: um coeficiente ajustado, uma previsão, uma probabilidade. A próxima seção volta à pergunta que abre o capítulo: o que significa estimar uma função a partir de dado, e o que muda entre prever e explicar.

## Estimar f: Predição e Inferência

> **📌 Nota**
>
> Esta seção corresponde à seção 2.1 de James et al. (2023).

A seção anterior fechou perguntando o que significa estimar uma função a partir de dado, e o que muda entre prever e explicar. É essa pergunta que abre o ISLP, com o exemplo que também abre esta seção: `Advertising`, o investimento em TV, rádio e jornal de duzentos mercados, contra as vendas de cada um.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("estilo-figuras.mplstyle")

### A formulação: Y = f(X) + ε

In [ ]:
propaganda = pd.read_csv("dados/Advertising.csv")
propaganda.shape, propaganda.columns.tolist()

Duzentos mercados e quatro colunas: `tv`, `radio` e `jornal`, o investimento em cada mídia, e `vendas`, o volume vendido.

Em notação, `X` = (tv, rádio, jornal) são os preditores — o que se mede e, em alguma medida, se controla —, e `Y` = vendas é a resposta, o que se quer prever ou explicar a partir de `X`. A suposição central deste livro inteiro é que existe uma relação entre os dois, e que ela se escreve como

$$Y = f(X) + \epsilon$$

`f` é uma função fixa, mas desconhecida, de `X`: é a informação sistemática que `X` carrega sobre `Y` — o que aconteceria, em média, se todo mercado com o mesmo investimento em propaganda vendesse a mesma quantidade. ε é o erro: tudo que influencia `Y` e não está em `X` — sazonalidade que a tabela não registra, um concorrente que baixou preço na mesma semana, o próprio ruído de medir "vendas" —, e por definição não pode ser previsto a partir de `X`. ε não é descuido de quem coletou o dado: é a distância entre o que `X` explica e tudo o mais que também importa. Por construção, ε é independente de `X` e tem média zero — não há tendência sistemática de errar para cima nem para baixo, só a parte que `X` não alcança.

### Prever ou explicar: duas perguntas diferentes

Uma vez que existe `f`, o que se faz com ela depende da pergunta. "Quanto vou vender com um orçamento assim?" é uma pergunta de **predição**: o que importa é acertar o valor de `Y`, e a forma exata de `f` pode ficar escondida — um modelo tratado como caixa-preta serve, desde que a previsão saia certa. "O que faz vender mais?" é uma pergunta de **inferência**: aqui a forma de `f` é o que se quer, porque a resposta está nela — quais mídias têm associação com as vendas, se o efeito é positivo ou negativo, se um investimento pequeno em jornal já esgota o que jornal tem a contribuir. Nesse sentido, "inferência" aqui é sobre *entender* a relação entre `X` e `Y`, não sobre calcular um erro-padrão ou um valor-p — essa segunda coisa é assunto de outra disciplina.

Um modelo pode responder bem a uma das duas perguntas e mal à outra, mesmo sobre o mesmo par (`X`, `Y`): uma caixa-preta pode prever vendas com folga e não dizer nada sobre qual mídia cortar num corte de orçamento; um modelo simples o bastante para se ler o efeito de cada mídia pode prever pior do que um mais flexível. Prever e explicar não são a mesma competência.

### Erro redutível e irredutível: o teto de qualquer modelo

Nenhuma estimativa `f̂` acerta `f` perfeitamente, e essa distância chama-se erro **redutível** — redutível porque escolher um método melhor, mais dado ou mais preditores pode encolhê-la. Mas mesmo que `f̂` fosse `f`, ponto a ponto, prever `Y` a partir de `X` ainda erraria: `Y` também depende de ε, que por definição não pode ser previsto a partir de `X`. Esse é o erro **irredutível**, e a distinção fica explícita ao decompor o erro quadrático médio da previsão `Ŷ = f̂(X)`:

$$
E\left[(Y - \hat{Y})^2\right] = \underbrace{\left[f(X) - \hat{f}(X)\right]^2}_{\text{redutível}} + \underbrace{\mathrm{Var}(\epsilon)}_{\text{irredutível}}
$$

O primeiro termo é o que um modelo melhor reduz. O segundo não muda com o modelo — é propriedade do problema, não do método — e por isso funciona como um teto: nenhum ajuste, por melhor que seja, empurra o erro esperado abaixo dele. Na prática o irredutível quase nunca é conhecido de antemão; mais adiante neste capítulo ele volta como o piso que explica por que a curva de erro de teste nunca chega a zero.

### Income1: um f verdadeiro, porque o dado é simulado

Na prática, `f` nunca aparece — só o dado observado, gerado por uma `f` desconhecida mais ε. `Income1` é a exceção que o ISLP usa de propósito: os pontos não vêm de pesquisa real, foram simulados pelos próprios autores, então a função que os gerou é conhecida por construção. É isso que torna possível desenhar `f` ao lado dos pontos, e não só os pontos.

In [ ]:
estudo_renda = pd.read_csv("dados/Income1.csv")
estudo_renda.shape, estudo_renda.columns.tolist()

Trinta indivíduos, anos de estudo (`escolaridade`) e renda em milhares de dólares (`renda`).

A função exata que os autores usaram para simular esses pontos não está disponível fora do pacote `ISLP` do R, que este material não instala. A curva a seguir não é aquela função: é uma média móvel sobre os pontos, ordenados por anos de estudo — uma estimativa suave de `f`, boa o bastante para mostrar a forma da relação, mas uma estimativa, não a verdade que gerou o dado.

In [ ]:
# Figura: Renda contra anos de estudo em `Income1`. A curva é uma estimativa suave de f; cada segmento liga um ponto observado à curva, e esse segmento é o ε daquela observação.
educacao = estudo_renda["escolaridade"].to_numpy()
renda_observada = estudo_renda["renda"].to_numpy()
f_suave = estudo_renda["renda"].rolling(window=13, center=True, min_periods=1).mean().to_numpy()

fig, ax = plt.subplots()
ax.vlines(
    educacao,
    np.minimum(renda_observada, f_suave),
    np.maximum(renda_observada, f_suave),
    color="C1",
    linewidth=1,
)
ax.plot(educacao, f_suave, color="C0", linewidth=2, label="f (estimativa suave)")
ax.scatter(educacao, renda_observada, color="C2", zorder=3, label="observado")
ax.set_xlabel("anos de estudo")
ax.set_ylabel("renda (milhares de dólares)")
ax.legend()
plt.tight_layout()
plt.show()

Os segmentos são o ε de cada observação: a distância entre o que a curva estima para aqueles anos de estudo e o que aquele indivíduo de fato ganha.

In [ ]:
residuo = renda_observada - f_suave
acima = int((residuo > 0).sum())
abaixo = int((residuo < 0).sum())
media = round(float(residuo.mean()), 2)
acima, abaixo, media

Dezesseis pontos ficam acima da curva e catorze abaixo, com a média dos resíduos em 0,10 — perto de zero, não exatamente zero, porque a curva usada aqui é uma estimativa de `f`, não a função que de fato gerou o dado. Nenhum modelo apaga esses segmentos: mesmo com a `f` exata em mãos — o caso raro que a simulação permite —, prever a renda de um indivíduo específico ainda erraria pelo tamanho do seu ε. É esse piso, e não uma falha de ajuste, que a próxima seção começa a explorar ao perguntar como, afinal, se estima `f`.

## Paramétrico e Não Paramétrico

> **📌 Nota**
>
> Esta seção corresponde à seção 2.1.2 de James et al. (2023).

A seção anterior fechou perguntando como, afinal, se estima `f` — a partir só do que se observa, sem acesso à função que gerou o dado. Há duas respostas bem diferentes para essa pergunta, e a diferença entre elas está inteira numa escolha que se faz antes de olhar qualquer ponto: assumir uma forma para `f`, ou deixar o dado decidir a forma sozinho.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.neighbors import KNeighborsRegressor

plt.style.use("estilo-figuras.mplstyle")

### O caminho paramétrico: assumir uma forma

O caminho paramétrico começa assumindo uma forma para `f` — uma reta, um plano, qualquer função que se escreva com um número fixo e pequeno de parâmetros — e troca o problema de estimar uma função inteira, livre para assumir qualquer formato, pelo problema bem mais simples de estimar esses poucos números. A vantagem é direta: poucos parâmetros pedem pouco dado, porque cada ponto observado ajuda a fixar todos eles ao mesmo tempo. O risco mora na mesma decisão: se a forma escolhida não é parecida com a `f` verdadeira, nenhuma quantidade de dado, nem cuidado nenhum no ajuste, conserta — o erro já está embutido na forma, antes mesmo de o primeiro ponto entrar na conta.

`Income2`, outro conjunto simulado do ISLP, dá o exemplo: a renda de trinta pessoas contra dois preditores, não um só como em `Income1` na seção anterior.

In [ ]:
renda2 = pd.read_csv("dados/Income2.csv")
renda2.shape, renda2.columns.tolist()

Trinta linhas, três colunas: `escolaridade` e `senioridade` são os preditores, `renda` é a resposta, em milhares de dólares. Com dois preditores, a forma paramétrica mais simples deixa de ser uma reta e passa a ser um plano — renda como combinação linear de escolaridade e senioridade.

In [ ]:
X = renda2[["escolaridade", "senioridade"]]
y = renda2["renda"]

plano = LinearRegression().fit(X, y)
[round(float(coeficiente), 3) for coeficiente in plano.coef_], round(float(plano.intercept_), 2)

O `LinearRegression` do `scikit-learn` ajusta exatamente essa forma: um plano com coeficiente 5,896 para escolaridade, 0,173 para senioridade, e intercepto -50,09. Três números — só três — descrevem a superfície inteira. Não importa se o dado tivesse trinta pontos ou trinta mil: mais dado deixaria a estimativa mais precisa, mas não mudaria quantos parâmetros o modelo tem.

### O caminho não paramétrico: deixar o dado dizer

O caminho não paramétrico não assume forma nenhuma para `f`: em vez de resumir a relação em poucos números, deixa a vizinhança de cada ponto decidir o valor previsto ali. *k* vizinhos mais próximos (k-NN) é o exemplo mais simples da família: para prever a renda de alguém com uma combinação de escolaridade e senioridade, ele procura as pessoas mais parecidas no dado observado e tira a média das rendas delas — sem supor que a relação seja um plano, uma curva suave ou qualquer coisa com nome fechado. A vantagem é acertar formas que o plano erraria de largada, porque nada na conta impõe retidão a `f`; o custo é precisar de muito mais dado para que essa liberdade compense e, se a flexibilidade for longe demais, ajustar não à relação verdadeira, mas ao ruído específico das trinta pessoas que compõem esta amostra.

In [ ]:
X.agg(["mean", "std"]).round(1)

Escolaridade tem média 16,4 anos e desvio padrão 3,8; senioridade, média 93,9 e desvio padrão 55,7 — uma dispersão bem maior, na mesma unidade de anos. k-NN decide "vizinho mais próximo" por distância, e com as duas colunas nessa escala tão diferente, a distância sem ajuste seria dominada pela que varia mais — a senioridade —, deixando a escolaridade quase sem voz na conta. A padronização resolve isso subtraindo a média e dividindo pelo desvio padrão de cada coluna:

In [ ]:
padronizado = (X - X.mean()) / X.std()
flexivel = KNeighborsRegressor(n_neighbors=3).fit(padronizado, y)

mse_plano = mean_squared_error(y, plano.predict(X))
mse_flexivel = mean_squared_error(y, flexivel.predict(padronizado))

erro_treino = pd.Series(
    {"plano": mse_plano, "k-NN flexível (k=3)": mse_flexivel}
).sort_values()
erro_treino.round(2)

Sobre os mesmos trinta pontos que o ajustaram, o k-NN flexível erra menos que o plano: 36,69 de erro quadrático médio contra 46,48. A superfície que não assume forma nenhuma encosta mais perto do dado observado do que o plano, rígido, consegue chegar — é exatamente a vantagem que a flexibilidade promete.

Só que a mesma flexibilidade que aproxima o ajuste do dado observado pode ir longe demais. Levando o k-NN ao extremo — prever com um único vizinho, em vez da média de três:

In [ ]:
memoriza = KNeighborsRegressor(n_neighbors=1).fit(padronizado, y)
round(mean_squared_error(y, memoriza.predict(padronizado)), 2)

o erro sobre o próprio dado de treino cai a exatamente zero. Faz sentido: com um vizinho só, a renda prevista para cada pessoa é a renda daquela mesma pessoa — ela é o seu próprio vizinho mais próximo, a distância zero. O modelo não aprendeu nada sobre como escolaridade e senioridade se relacionam com renda; decorou as trinta respostas que já tinha. Essa armadilha tem nome — *overfitting* —, e é o preço que a flexibilidade cobra quando nada a segura: ajustar tão perto do dado observado que o modelo passa a repetir o ruído daquela amostra específica, em vez da relação que a gerou. A seção 7.6 mede isso a sério, contra dado que o modelo não viu durante o ajuste; aqui fica só o nome e o sintoma.

> **🔷 Conceito**
>
> | | Paramétrico | Não paramétrico |
> |---|---|---|
> | Forma de `f` | assumida antes de ver o dado (reta, plano, ...) | não assumida — o dado decide |
> | O que se estima | um número fixo de parâmetros | a superfície inteira |
> | Quanto dado exige | pouco | muito mais |
> | Risco principal | a forma errada, que nenhum dado corrige | decorar o ruído da amostra (*overfitting*) |

### A rigidez de um contra a flexibilidade do outro

A diferença fica mais clara em figura do que em número: as mesmas trinta pessoas, com a superfície que cada ajuste prevê para qualquer combinação de escolaridade e senioridade, não só as observadas. Os pontos da malha abaixo também precisam de padronização antes de passar pelo k-NN — e usam a média e o desvio padrão do próprio treino (`X.mean()`, `X.std()`), nunca recalculados sobre a malha, porque é essa régua, e só essa, que o ajuste aprendeu.

In [ ]:
# Figura: Renda prevista para toda combinação de escolaridade e senioridade em `Income2`, pelo plano paramétrico (esquerda) e pelo k-NN flexível com k=3 (direita). Os pontos são as trinta pessoas observadas. As faixas retas e paralelas do plano não têm como acompanhar cada aglomerado de pontos; a superfície flexível dobra ao redor deles.
grade_escolaridade = np.linspace(X["escolaridade"].min(), X["escolaridade"].max(), 60)
grade_senioridade = np.linspace(X["senioridade"].min(), X["senioridade"].max(), 60)
malha_e, malha_s = np.meshgrid(grade_escolaridade, grade_senioridade)
grade = pd.DataFrame({"escolaridade": malha_e.ravel(), "senioridade": malha_s.ravel()})
grade_padronizada = (grade - X.mean()) / X.std()

superficie_plano = plano.predict(grade).reshape(malha_e.shape)
superficie_flexivel = flexivel.predict(grade_padronizada).reshape(malha_e.shape)

niveis = np.linspace(
    min(superficie_plano.min(), superficie_flexivel.min()),
    max(superficie_plano.max(), superficie_flexivel.max()),
    13,
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))

ax1.contourf(malha_e, malha_s, superficie_plano, levels=niveis, cmap="Blues")
ax1.scatter(X["escolaridade"], X["senioridade"], color="C1", s=18, edgecolor="white", linewidth=0.6)
ax1.set_xlabel("escolaridade")
ax1.set_ylabel("senioridade")
ax1.set_title("paramétrico: o plano")

mapa = ax2.contourf(malha_e, malha_s, superficie_flexivel, levels=niveis, cmap="Blues")
ax2.scatter(X["escolaridade"], X["senioridade"], color="C1", s=18, edgecolor="white", linewidth=0.6)
ax2.set_xlabel("escolaridade")
ax2.set_ylabel("senioridade")
ax2.set_title("não paramétrico: k-NN (k=3)")

barra = fig.colorbar(mapa, ax=[ax1, ax2], shrink=0.85, pad=0.02)
barra.set_label("renda prevista")

plt.show()

O plano nunca se curva, e é exatamente por isso que ele erra pouco: três números não têm como acompanhar cada solavanco do dado, só a tendência geral. O k-NN se curva ao redor de cada aglomerado de pontos, e é por isso que ele decora — sem outra coisa dizendo até onde ir, nada o impede de seguir cada ponto isoladamente até o extremo que os números acima já mostraram. Essa flexibilidade tem outro preço, além do dado que ela exige: um modelo que se adapta tão de perto ao que observou fica mais difícil de ler — não sobra um coeficiente único para dizer "mais um ano de escolaridade vale tanto de renda". É esse compromisso, entre acertar mais e entender menos, que a próxima seção discute.

## Precisão contra Interpretabilidade

> **📌 Nota**
>
> Esta seção corresponde à seção 2.1.3 de James et al. (2023).

A seção anterior fechou apontando um preço que a flexibilidade cobra além do dado que ela exige: um modelo que se adapta tão de perto ao que observou fica mais difícil de ler — não sobra um coeficiente único para dizer quanto vale cada preditor. Esse preço não é acidente de um exemplo: é um padrão que atravessa o aprendizado estatístico inteiro, e vale a pena nomear com precisão antes de seguir adiante.

In [ ]:
import matplotlib.pyplot as plt

plt.style.use("estilo-figuras.mplstyle")

### O compromisso: acertar mais, entender menos

De um lado da escolha ficam os métodos rígidos: a regressão linear que a seção anterior ajustou a `Income2` reduz a superfície inteira a três números, e é justamente por ter tão pouco a ajustar que cada um deles se lê de cara — "mais um ano de escolaridade vale tanto de renda". Do outro lado ficam os métodos flexíveis: o k-NN que decorou as trinta pessoas não tem coeficiente nenhum para apontar, só uma vizinhança que muda de ponto a ponto; descrever o que ele "aprendeu" sobre a relação entre escolaridade, senioridade e renda exige desenhar a superfície inteira, não escrever uma frase.

O padrão geral é esse: quanto mais flexível o método, mais formas de `f` ele consegue acompanhar — e por isso costuma prever melhor —, e menos ele se deixa resumir em algo que uma pessoa lê e repete. Não é uma lei sem exceção, mas é forte o bastante para orientar a escolha de método antes mesmo de olhar o dado: acertar mais e entender menos costumam vir no mesmo pacote.

### Por que escolher o menos flexível de propósito

Se a flexibilidade costuma vir com mais precisão, por que alguém abriria mão dela? Duas razões, e nenhuma das duas é conservadorismo.

A primeira é a pergunta que se está fazendo. A seção anterior distinguiu predição de inferência: prever só pede que `Ŷ` saia perto de `Y`; inferir pede entender a forma de `f` — que preditor pesa mais, em que direção, com que força. Um método flexível pode prever tão bem quanto promete e ainda assim não servir para essa segunda pergunta: se a única forma de descrever o que ele fez é reproduzir o código inteiro, não há como apontar nele "isto é o que empurra a resposta para cima". Quando a pergunta é de inferência, um modelo que acerta e não se deixa explicar simplesmente não responde ao que foi perguntado — por melhor que seja a precisão dele.

A segunda é a quantidade de dado disponível. A seção anterior também mostrou o preço que a flexibilidade cobra em dado: pouco dado não basta para uma superfície livre aprender a relação verdadeira, e o que ela aprende no lugar é o ruído específico daquela amostra pequena — a armadilha que ali ficou batizada de *overfitting*. Com pouco dado, o método flexível não chega nem a entregar a precisão que prometia: erra menos no que já viu e mais no que ainda vai ver. Um método rígido, nesse caso, não é a segunda opção por cautela — pode ser a única que entrega alguma precisão de verdade.

> **🔷 Conceito**
>
> | Escolher o **menos** flexível de propósito | Por quê |
> |---|---|
> | A pergunta é de inferência | um ajuste que não se lê não responde "o que explica a resposta", mesmo acertando |
> | O dado é pouco | o método flexível não tem com que aprender a relação — decora a amostra em vez disso |

### O mapa: flexibilidade contra interpretabilidade

Colocando lado a lado alguns dos métodos que este material ainda vai ver — regressão linear, lasso, árvores de decisão, ensembles como bagging e boosting, k-vizinhos mais próximos e redes neurais —, o padrão desta seção aparece como duas pontas de um mesmo eixo: de um lado, métodos rígidos o bastante para se ler o efeito de cada preditor; do outro, métodos livres o bastante para acompanhar qualquer forma que o dado tenha.

Não há dado nenhum por trás do mapa a seguir — nenhum experimento mediu "flexibilidade" ou "interpretabilidade" em unidade nenhuma. É uma comparação relativa entre famílias de método, do jeito que a literatura de aprendizado estatístico costuma desenhar: útil para orientar uma escolha antes de ajustar qualquer coisa, não um número a se citar depois.

In [ ]:
# Figura: Flexibilidade contra interpretabilidade — um mapa qualitativo, sem unidade em nenhum dos dois eixos: a posição de cada método é uma comparação relativa entre famílias, não uma coordenada medida sobre dado nenhum.
metodos = {
    "lasso": (0.08, 0.92),
    "regressão linear": (0.22, 0.80),
    "árvore de decisão": (0.45, 0.55),
    "k-vizinhos\nmais próximos": (0.58, 0.38),
    "bagging e boosting": (0.72, 0.28),
    "redes neurais": (0.92, 0.10),
}

fig, ax = plt.subplots(figsize=(7, 5.2))
for nome, (flexibilidade, interpretabilidade) in metodos.items():
    ax.scatter(flexibilidade, interpretabilidade, color="C0", s=70, zorder=3)
    ax.annotate(
        nome,
        (flexibilidade, interpretabilidade),
        textcoords="offset points",
        xytext=(8, 6),
        fontsize=9,
    )

ax.set_xlim(0, 1.05)
ax.set_ylim(0, 1.05)
ax.set_xticks([0.05, 1.0])
ax.set_xticklabels(["baixa", "alta"])
ax.set_yticks([0.05, 1.0])
ax.set_yticklabels(["baixa", "alta"])
ax.set_xlabel("flexibilidade (ordem qualitativa, sem escala)")
ax.set_ylabel("interpretabilidade (ordem qualitativa, sem escala)")
plt.tight_layout()
plt.show()

Lasso e regressão linear ficam no canto rígido e legível, com poucos coeficientes cada. Redes neurais e ensembles como bagging e boosting ficam no canto oposto, onde mora a precisão que costumam entregar. Árvore de decisão e k-vizinhos mais próximos ficam no meio — nem tão fechados quanto uma reta, nem tão opacos quanto uma rede ou um ensemble —, e é essa posição relativa, não um valor fixo, que este material vai reencontrar quando cada método ganhar o espaço que merece.

Nenhum desses métodos é explicado aqui — cada um tem, à frente, o capítulo que lhe cabe. O que fica desta seção é a posição relativa e o motivo para levá-la a sério: antes de escolher qual construir, vale perguntar que pergunta se está fazendo e quanto dado se tem, porque as duas respostas empurram a escolha para lados opostos do mesmo mapa.

Flexibilidade e interpretabilidade não são o único par de eixos que decide qual método usar. Há uma pergunta anterior a essa: existe uma resposta para aprender contra, ou só um conjunto de preditores sem rótulo nenhum? É essa distinção — entre aprendizado supervisionado e não supervisionado — que a próxima seção faz.

## Supervisionado e Não Supervisionado

> **📌 Nota**
>
> Esta seção corresponde à seção 2.1.4 de James et al. (2023).

> **⚠️ Atenção — Em construção**
>
> O conteúdo desta seção ainda será escrito.

## Qualidade do Ajuste e o Compromisso Viés-Variância

> **📌 Nota**
>
> Esta seção corresponde à seção 2.2.1 de James et al. (2023).

> **⚠️ Atenção — Em construção**
>
> O conteúdo desta seção ainda será escrito.

## Classificação e o Classificador de Bayes

> **📌 Nota**
>
> Esta seção corresponde à seção 2.2.3 de James et al. (2023).

> **⚠️ Atenção — Em construção**
>
> O conteúdo desta seção ainda será escrito.

## Leituras adicionais

*A escrever.*

## Referências

- **James; Witten; Hastie; Tibshirani; Taylor**. *An Introduction to Statistical Learning with Applications in Python*. Springer. 2023.